In [1]:
import os
import pathlib
import sys


DNSCBOR_EVAL_DIR = pathlib.Path.cwd()
os.environ["DNSCBOR_EVAL_DIR"] = str(DNSCBOR_EVAL_DIR)

if str((DNSCBOR_EVAL_DIR / "..").absolute()) not in sys.path:
    sys.path.append(str((DNSCBOR_EVAL_DIR / "..").absolute()))

from utils import list_code

# application/dns+cbor: Dataset Collection

## MassDNS Tranco List at Public Resolvers

We use the [MassDNS] tool by Birk Blechschmidt and Quirin Scheitle to query AAAA, A, HTTPS, NS, PTR, DS, RRSIG, DNSKEY, NSEC, and NSEC3 records for the names in the Tranco list from the following public resolvers:

- `1.1.1.1` (CloudFlare)
- `8.8.8.8` (Google)
- `9.9.9.9` (Quad9)

To build MassDNS, use the following command.

[MassDNS]: https://github.com/blechschmidt/massdns

In [2]:
%%bash
git -C "${DNSCBOR_EVAL_DIR}"/.. submodule update --init
make -C "${DNSCBOR_EVAL_DIR}/massdns"

Submodule 'massdns' (https://github.com/blechschmidt/massdns.git) registered for path '04_cbor4dns_eval/massdns'
Cloning into '/home/vagrant/cbor-dns-eval-tbd/04_cbor4dns_eval/massdns'...


Submodule path '04_cbor4dns_eval/massdns': checked out 'bad45b873057637ae69b9d9f4c1c179126f97a48'
make: Entering directory '/home/vagrant/cbor-dns-eval-tbd/04_cbor4dns_eval/massdns'
mkdir -p bin
cc    -DMASSDNS_REVISION=\"v1.1.0-7-gbad45b8\" -O3 -std=c11 -DHAVE_EPOLL -DHAVE_SYSINFO -Wall -fstack-protector-strong src/main.c -o bin/massdns
make: Leaving directory '/home/vagrant/cbor-dns-eval-tbd/04_cbor4dns_eval/massdns'


We use the following to generate the DNS traffic. **This may take run for a while and may use up your bandwidth significantly; we recommend running it off-site or while you are asleep.** We provide a complete PCAP at TBD if you do not have the resources to generate it.

In [3]:
list_code(DNSCBOR_EVAL_DIR / "massdns_resolve.sh")

#!/bin/bash

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"

EXP="${EXP:-}"
MASSDNS_DIR="${SCRIPT_DIR}/massdns"
RESULTS_DIR="${RESULTS_DIR:-${SCRIPT_DIR}/input_datasets/tranco}"
PROCS="$(grep -c '^processor' /proc/cpuinfo)"
TRANCO_SET=KJ49W
RESOLVERS=( "8.8.8.8" "1.1.1.1" "9.9.9.9" )
CNAME_ITERATION=1
RRTYPES="-t AAAA -t A -t HTTPS -t NS -t PTR -t DS -t RRSIG -t DNSKEY -t NSEC -t NSEC3"
RRTYPES_NAMES=$(echo "${RRTYPES}" | sed -E 's/ ?-t /_/g')

if [ $# -gt 0 ]; then
    SNIFF_IFACE=$1
fi

mkdir -p "${RESULTS_DIR}"

if [ ! -f "${RESULTS_DIR}/tranco_${TRANCO_SET}_full.csv" ]; then
    wget -O "${RESULTS_DIR}/tranco_${TRANCO_SET}_full.csv" "https://tranco-list.eu/download/${TRANCO_SET}/full" \
        2>&1 || exit 1
fi

echo -n "" > "${SCRIPT_DIR}/resolvers.txt"

for RESOLVER in ${RESOLVERS[@]}; do
    echo "${RESOLVER}" >> "${SCRIPT_DIR}/resolvers.txt"
done

if [ -n "${SNIFF_IFACE}" ]; then
    tshark -i "${SNIFF_IFACE}" \
        -w "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}.pcapng" "port 53" &
    TSHARK_PID=$!
    sleep 5  # make sure tshark listens
fi

awk -F, '{print $2}' "${RESULTS_DIR}/tranco_${TRANCO_SET}_full.csv" | sed 's/\r$//' |
    "${MASSDNS_DIR}/bin/massdns" --status-format json -r "${SCRIPT_DIR}/resolvers.txt" ${RRTYPES} -o J | \
    sed "s/}\$/,\"cname_iteration\":${CNAME_ITERATION}}/" \
    > "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_${CNAME_ITERATION}.ndjson"

while grep -q -e "CNAME" \
        "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_${CNAME_ITERATION}.ndjson"; do
     CNAME_ITERATION=$((CNAME_ITERATION + 1))
    cat "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_$(( CNAME_ITERATION - 1 )).ndjson" | \
    jq -r '.data[][] | select (.type == "CNAME") | .data' | sort -u | \
        "${MASSDNS_DIR}/bin/massdns" --status-format json -r "${SCRIPT_DIR}/resolvers.txt" ${RRTYPES} -o J | \
    sed "s/}\$/,\"cname_iteration\":${CNAME_ITERATION}}/" \
        > "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_${CNAME_ITERATION}.ndjson" \

    if [ ${CNAME_ITERATION} -eq 10 ]; then
         break
    fi
done

if [ -n "${TSHARK_PID}" ]; then
    kill "${TSHARK_PID}"
fi

cat "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}_"*.ndjson | xz \
    > "${RESULTS_DIR}/tranco_${TRANCO_SET}_full_${RRTYPES_NAMES}${EXP}.ndjson.xz"

You need the network interface you want to query over. In the Vagrant VM, this is normally `eth0`. Use the following command to find it. The one that is **not** `lo`, is the one you want.

In [4]:
%%bash
ip link

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN mode DEFAULT group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UP mode DEFAULT group default qlen 1000
    link/ether 08:00:27:e8:48:7c brd ff:ff:ff:ff:ff:ff
    altname enp0s3


In [5]:
%%bash
tmux new-session -s "massdns" -d "'${DNSCBOR_EVAL_DIR}'/massdns_resolve.sh eth0 2> '${DNSCBOR_EVAL_DIR}'/input_datasets/tranco/tranco_KJ49W_full__AAAA_A_HTTPS_NS_PTR_DS_RRSIG_DNSKEY_NSEC_NSEC3.status.txt"

You can follow this `tmux` session with the `tmux attach -t massdns` command. To kill the tmux session prematurely, use

```sh
tmux send-keys -t "massdns" C-c
```

Errors are logged within `04_cbor4dns_eval/input_datasets/tranco/tranco_KJ49W_full__AAAA_A_HTTPS_NS_PTR_DS_RRSIG_DNSKEY_NSEC_NSEC3.status.txt`

## Download IoT Datasets

This notebook expects the IoT data sets to be present in the `./04_cbor4dns_eval/input_dataset` directory. The following sub-directories are expected for the datasets of the IoTFinder, MonIoTr, and Yourthings studies, respectively.

- `iotfinder`
- `moniotr`
- `yourthings`

Please read below how to acquire them.

### Download IoTFinder & YourThings Data

You can freely download the datasets IoTFinder and YourThings datasets at https://yourthings.info/data/. The authors ask kindly to cite their work if you use it in your published data.

1. Roberto Perdisci, Thomas Papastergiou, Omar Alrawi, Manos Antonakakis; [IoTFinder: Efficient Large-Scale Identification of IoT Devices via Passive DNS Traffic Analysis](https://doi.org/10.1109/EuroSP48549.2020.00037), in IEEE European Symposium of Security & Privacy, Sept 2020.
2. Omar Alrawi, Chaz Lever, Manos Antonakakis, Fabian Monrose; [SoK: Security Evaluation of Home-Based IoT Deployments](https://doi.org/10.1109/SP.2019.00013), in IEEE Symposium of Security & Privacy, May 2019.

For your convenience, we provided the following script to download the datasets.
Keep in mind that this downloads about 140 Gbytes of compressed data and may thus, depending on your Internet connection, take several hours complete. Over a 50 MBit/s connection it took about 7 hours. We recommend to run it in background, e.g., in a `tmux` session.

In [6]:
list_code(DNSCBOR_EVAL_DIR / "download_iotfinder_and_yourthings.sh")

#!/usr/bin/env bash
#
# collect_dns_hex.sh
# Copyright (C) 2023 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR="$( cd -- "$( dirname -- "${BASH_SOURCE[0]}" )" &> /dev/null && pwd )"
INPUT_DATASETS="${SCRIPT_DIR}/input_datasets"

IOTFINDER_URLS=(
    "https://www.dropbox.com/s/3tv44ywzyahy3km/dns_2019_08.tgz?dl=0"
    "https://www.dropbox.com/s/v5pga0qpnywe1et/dns_2019_09.tgz?dl=0"
)
YOURTHINGS_URLS=(
    "https://www.dropbox.com/s/0jltu5o4a9kk7z8/iot_traffic20180320.tgz?dl=0"
    "https://www.dropbox.com/s/xo75oan8juoyb9o/iot_traffic20180321.tgz?dl=0"
    "https://www.dropbox.com/s/2iolbgnaw68cpb0/iot_traffic20180328.tgz?dl=0"
    "https://www.dropbox.com/s/blyem2vfbwbwd4h/iot_traffic20180410.tgz?dl=0"
    "https://www.dropbox.com/s/rcodps77sot88xl/iot_traffic20180411.tgz?dl=0"
    "https://www.dropbox.com/s/ldpkjs799wxf7la/iot_traffic20180412.tgz?dl=0"
    "https://www.dropbox.com/s/z11dgm9u6kuzvml/iot_traffic20180413.tgz?dl=0"
    "https://www.dropbox.com/s/2j62g8qxhby4sqh/iot_traffic20180414.tgz?dl=0"
    "https://www.dropbox.com/s/n6epxutlemcrcrp/iot_traffic20180415.tgz?dl=0"
    "https://www.dropbox.com/s/tfrhq7noobgxpi0/iot_traffic20180416.tgz?dl=0"
    "https://www.dropbox.com/s/gsly960mzi6f0on/iot_traffic20180417.tgz?dl=0"
    "https://www.dropbox.com/s/8klt5f9fr164f5n/iot_traffic20180418.tgz?dl=0"
    "https://www.dropbox.com/s/c0uxli3cirzill1/iot_traffic20180419.tgz?dl=0"
)

download_dataset() {
    name="${1}"
    urls=(${@})
    unset urls[0]

    mkdir -p "${INPUT_DATASETS}/${name}"
    for url in ${urls[@]}; do
        curl -L "$url" | tar -xz -C "${INPUT_DATASETS}/${name}"
    done
}

download_dataset "iotfinder" "${IOTFINDER_URLS[@]}"
download_dataset "yourthings" "${YOURTHINGS_URLS[@]}"

In [7]:
%%bash
tmux new-session -s "download_iot" -d "${DNSCBOR_EVAL_DIR}/download_iotfinder_and_yourthings.sh"

You can follow this `tmux` session with the `tmux attach -t download_iot` command. To kill the tmux session prematurely, use

```sh
tmux send-keys -t "download_iot" C-c
```

### Download MonIoTr

The authors of the MonIoTr study ask you to agree to their data sharing agreement to download their data set. Please see [their website](https://moniotrlab.khoury.northeastern.edu/publications/imc19/) for more details. Once you received the link to their drive, download the `iot-data.tgz` their and drop the unpacked files in `./04_cbor4dns_eval/input_dataset/moniotr` manually (with `DNSCBOR_EVAL_DIR` set to `./04_cbor4dns_eval`).

```sh
mkdir -p ${DNSCBOR_EVAL_DIR}/input_datasets/moniotr
tar -C ${DNSCBOR_EVAL_DIR}/input_datasets/moniotr -xzf "<path to downloaded iot-data.tgz>"
```

### Checking Downloaded Datsets

Now, all datasets should be in `./04_cbor4dns_eval/input_dataset`. You can check the correctness with the following command.

In [8]:
!ls "{DNSCBOR_EVAL_DIR}/input_datasets/"
!cd "{DNSCBOR_EVAL_DIR}/input_datasets/" && sha256sum --status -c input_datasets.sha256sum && echo "Input datasets OK" || echo "Error checking dataset consistency" >&2

input_datasets.sha256sum  iotfinder  moniotr  yourthings
Input datasets OK
